# Halo builder

This notebook has two stages:

1. construct the initial (t=0) halo and inspect its density projection;
2. evolve the same initial halo in time.

Run the cells from top to bottom. The single Convention object in the first
code cell is passed to fdmtools.FDMTools and is the source of all physical
constants and unit conversions.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import scipy.interpolate as scinterp
import fdmtools

warnings.filterwarnings('ignore')
plt.rc('font', size=14)

# Keep one physical convention for both the notebook and fdmtools.py.
convention = fdmtools.Convention()

# =============================
# Runtime configuration
# =============================
have_soliton = True
m_as = [1.0e-22]
Nmaxes = [None]
seeds = np.arange(2)

# Halo / profile setup
r_setup_res = 512
a = 10.0
M = 1.0e9
R_max = 200.0
R_fit = 80.0

# Time setup
t_step = 0.1
t_stop = 11.0
t_array = np.arange(start=0.0, stop=t_stop, step=t_step)

# Halo reconstruction and plotting
plot_seed = int(seeds[0])
projection_rmax = 30.0
projection_resolution = 192
projection_rshells = 128

## Part I — construct the t=0 halo

The loop below only builds the profile, potential, eigenmodes, amplitudes, and
random phases. It does not create any time-dependent snapshots yet.

In [ ]:
@dataclass
class HaloState:
    fdm: object
    m_a: float
    Nmax: object
    seed: int
    folder: Path
    prefix: Path
    rho_in: object
    phi: object
    rho: object
    M_tot: float
    E_nl: np.ndarray
    R_nl: np.ndarray
    a_nlm: np.ndarray
    phase_nlm: np.ndarray


def make_profile(fdm):
    """Build the NFW/soliton target profile and its initial potential."""
    r_setup = np.logspace(-3, 3, r_setup_res)
    rho_nfw = M / (2.0 * np.pi * a**3) / (r_setup / a) / (1.0 + r_setup / a)**3

    if have_soliton:
        r_core = fdm.r_core(M)
        rho_soliton = fdm.rho_core(r_setup, r_core)
        core_indices = np.flatnonzero(rho_soliton > rho_nfw)
        if core_indices.size == 0:
            raise ValueError('The soliton profile never exceeds the NFW profile.')
        transition = core_indices[-1] + 1
        rho_values = np.concatenate((rho_soliton[:transition], rho_nfw[transition:]))
        print(f'Using soliton core: rc={r_core:.3f} kpc, rho_c(0)={rho_soliton[0]:.3e}')
    else:
        rho_values = rho_nfw
        print('Using NFW profile without soliton core')

    rho_in = scinterp.CubicSpline(r_setup, rho_values)
    phi, rho, M_tot = fdm.initialize_potential(rho_in, core=False, r_max=R_max)
    fE = fdm.DF_invert(phi, rho)
    return rho_in, phi, rho, fE, M_tot


def apply_mode_cut(a_nlm, Nmax):
    """Return a copy of amplitudes with optional principal-number truncation."""
    amplitudes = np.array(a_nlm, copy=True)
    if Nmax is not None:
        n_idx = np.arange(amplitudes.shape[0])[:, None]
        l_idx = np.arange(amplitudes.shape[1])[None, :]
        amplitudes[n_idx + l_idx + 1 > Nmax] = 0.0
    return amplitudes


def make_output_folder(m_a, Nmax):
    suffix = '' if have_soliton else '_nosoliton'
    folder = Path(f'ma{m_a:.1e}M{M:.1e}ts{t_step}rres{r_setup_res}{suffix}')
    if Nmax is not None:
        folder = Path(f'{folder}_Nmax{Nmax}')
    return folder


halo_states = []

for Nmax in Nmaxes:
    for m_a in m_as:
        fdm = fdmtools.FDMTools(m_a=m_a, convention=convention)
        print(f'\nma={m_a:.3e}')
        rho_in, phi, rho, fE, M_tot = make_profile(fdm)
        print(f'M_tot={M_tot:.3e} M_sun')
        folder = make_output_folder(m_a, Nmax)

        for seed in seeds:
            seed = int(seed)
            print(f'\nBuilding seed {seed}')
            E_nl, R_nl, a_nlm, phase_nlm, _, _ = fdm.solve_amps_itr(
                R_max,
                R_fit,
                phi,
                rho,
                M,
                fE,
                maxiter=1,
                method='Isotropic',
                bins=50,
                schwarz_iter=2500,
                seed=seed,
                plot_progress=False,
            )
            a_nlm = apply_mode_cut(a_nlm, Nmax)

            prefix = folder / f's{seed}'
            prefix.mkdir(parents=True, exist_ok=True)
            nmax_tag = 'all' if Nmax is None else str(Nmax)

            r_profile = np.logspace(-2, np.log10(R_fit), 200)
            den_base_profile, _ = fdm.den_from_amps(r_profile, a_nlm, R_nl)
            if seed == plot_seed:
                fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
                ax.loglog(r_profile, rho_in(r_profile), 'k--', label='Target')
                ax.loglog(r_profile, den_base_profile, 'b-', label='Halo from amplitudes')
                ax.set_xlim(0.01, R_fit)
                ax.set_xlabel(r'$r$ [kpc]')
                ax.set_ylabel(r'$\rho$ [$M_\odot$ / kpc$^3$]')
                ax.set_title(f'Nmax={Nmax}, seed={seed}')
                ax.legend()
                fig.savefig(prefix / f'density_profile_n{nmax_tag}.png', dpi=150)
                plt.show()
                plt.close(fig)

            valid_nl = E_nl != 0.0
            l_values = np.arange(a_nlm.shape[1])
            total_modes = int(np.sum(valid_nl * (2 * l_values[None, :] + 1)))
            active_modes = int(np.sum((valid_nl & (a_nlm != 0.0)) * (2 * l_values[None, :] + 1)))
            print(f'Seed {seed}: total modes={total_modes}, active modes={active_modes}')

            halo_states.append(
                HaloState(
                    fdm=fdm,
                    m_a=m_a,
                    Nmax=Nmax,
                    seed=seed,
                    folder=folder,
                    prefix=prefix,
                    rho_in=rho_in,
                    phi=phi,
                    rho=rho,
                    M_tot=M_tot,
                    E_nl=E_nl,
                    R_nl=R_nl,
                    a_nlm=a_nlm,
                    phase_nlm=phase_nlm,
                )
            )

print(f'\nConstructed {len(halo_states)} initial halo(s).')

### Initial density viewed along the z axis

This is a line-of-sight projection, \(\Sigma(x,y)=\int
\rho(x,y,z)\,dz\),
of the first constructed halo at t=0.

In [ ]:
def plot_initial_density_projection(state, rmax=30.0, resolution=192, rshell_count=128):
    """Plot and return the t=0 surface density projected along the z axis."""
    rshells = np.linspace(1.0e-4, rmax, rshell_count)
    psi_shells, Rs, thetas, phis = state.fdm.build_halo(
        rshells,
        state.E_nl,
        state.R_nl,
        state.a_nlm,
        state.phase_nlm,
    )
    density_shells = np.abs(psi_shells)**2
    density_cube = state.fdm.halo_to_cube(
        density_shells,
        Rs,
        thetas,
        phis,
        res=resolution,
        rmax=rmax,
    )

    axis = np.linspace(-rmax, rmax, resolution)
    surface_density = np.trapz(density_cube, x=axis, axis=2)
    surface_density = np.maximum(np.real(surface_density), np.finfo(float).tiny)
    log_surface_density = np.log10(surface_density)

    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
    image = ax.imshow(
        log_surface_density.T,
        origin='lower',
        extent=[-rmax, rmax, -rmax, rmax],
        aspect='equal',
        cmap='magma',
        vmin=np.percentile(log_surface_density, 1),
        vmax=np.percentile(log_surface_density, 99),
    )
    ax.set_xlabel(r'$x$ [kpc]')
    ax.set_ylabel(r'$y$ [kpc]')
    ax.set_title(f't=0 density projected along z (seed={state.seed})')
    fig.colorbar(image, ax=ax, label=r'$\log_{10}\Sigma$ [$M_\odot$ / kpc$^2$]')
    fig.savefig(state.prefix / 'density_projection_z_t0.png', dpi=150)
    plt.show()
    plt.close(fig)
    return surface_density


state_to_plot = halo_states[0]
t0_surface_density = plot_initial_density_projection(
    state_to_plot,
    rmax=projection_rmax,
    resolution=projection_resolution,
    rshell_count=projection_rshells,
)

## Part II — evolve the initial halo

The following cell advances the eigenmode phases and reconstructs the halo at
a requested time. It keeps the evolved wavefunction in memory and does not
write an external potential format.

In [ ]:
def evolve_halo(state, time, rmax=30.0, rshell_count=128):
    """Reconstruct a halo after evolving its eigenmode phases by time Gyr."""
    rshells = np.linspace(1.0e-4, rmax, rshell_count)
    return state.fdm.evolve(
        rshells,
        state.E_nl,
        state.R_nl,
        state.a_nlm,
        state.phase_nlm,
        time,
    )


state_to_evolve = halo_states[0]
evolution_time = float(t_array[-1])
evolved_psi, evolved_Rs, evolved_thetas, evolved_phis = evolve_halo(
    state_to_evolve,
    evolution_time,
    rmax=projection_rmax,
    rshell_count=projection_rshells,
)
evolved_density = np.abs(evolved_psi)**2
print(f'Evolved seed {state_to_evolve.seed} to t={evolution_time:.2f} Gyr.')